In [1]:
from segmentiq.builder import *
from segmentiq.scorer import *
from segmentiq.utils.data_loader import *

# Example Feature Groupings Setup
# business_groups = {
# "delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m", "max_dpd_12m"],            
# "transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
# "spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"],
# "repayment_vars":["payment_ratio_avg_3m", "payment_ratio_avg_6m", "payment_ratio_avg_12m"],
# "card_utilization_vars":["utilization_avg_3m", "utilization_avg_6m", "utilization_avg_12m", "utilization_max_12m"],
# }

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000, 5000, 1000, 500],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="Default_Status",
    n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.5,
    min_events = 100,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=False,
    max_feature_reuse=2,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=None, #business_groups,
    ignore_features=None,
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"credit_card_default_2.5M.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-11 16:59:17,375 | INFO     | [builder.py:335] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-11 16:59:17,648 | INFO     | [builder.py:350] | Dynamic Grid Search Enabled: 8 total configurations.
2026-07-11 16:59:17,651 | INFO     | [builder.py:371] | Iteration 1 | Remaining Volume: 250,000 | Base Rate: 26.02%
2026-07-11 16:59:18,726 | INFO     | [builder.py:507] | Feature Usage Tracker Update -> 'Missed_Payment_Pattern' used count = 1
2026-07-11 16:59:18,727 | INFO     | [builder.py:522] | Segment 1 Captured (Size Floor: 10000 | Lift Floor: 1.5): Missed_Payment_Pattern IN ('Frequent')
2026-07-11 16:59:18,981 | INFO     | [builder.py:371] | Iteration 2 | Remaining Volume: 212,468 | Base Rate: 21.94%
2026-07-11 16:59:23,541 | INFO     | [builder.py:507] | Feature Usage Tracker Update -> 'Prior_Bankruptcy' used count = 1
2026-07-11 16:59:23,541 | INFO     | [builder.py:507] | Feature Usage Tracker Update -> 'Housing_Status' used count = 1
2026-07-11 16:59:23,542 | INFO  

In [2]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |     capture_rate    |        lift        |
+---------+-------------+---------------+--------------------+--------------------+---------------------+--------------------+
|   0.0   |   117652.0  |    15465.0    | 13.144697922687246 | 26.024399999999996 |       47.0608       | 0.5050912959640663 |
|   1.0   |   37532.0   |    18437.0    | 49.123414686134495 | 26.024399999999996 |       15.0128       | 1.8875906720667721 |
|   2.0   |    743.0    |     721.0     | 97.03903095558546  | 26.024399999999996 | 0.29719999999999996 | 3.7287711130933077 |
|   3.0   |    5567.0   |     5032.0    | 90.38979701814263  | 26.024399999999996 |        2.2268       | 3.473271123182192  |
|   4.0   |   24940.0   |     7999.0    | 32.07297514033681  | 26.024399999999996 |        9.976        | 1.232

In [3]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str
SQL Filter: Missed_Payment_Pattern IN ('Frequent')
--------------------------------------------------
Segment ID: 2
Raw Rule:   Prior_Bankruptcy=<ArrowStringArray>
['Yes']
Length: 1, dtype: str & Housing_Status=<ArrowStringArray>
['Rent']
Length: 1, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str
SQL Filter: Prior_Bankruptcy IN ('Yes') AND Housing_Status IN ('Rent') AND Employment_Type IN ('Unemployed')
--------------------------------------------------
Segment ID: 3
Raw Rule:   Housing_Status=<ArrowStringArray>
['Rent']
Length: 1, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str
SQL Filter: Housing_Status IN ('Rent') AND Employment_Type IN ('Unemployed')
--------------------------------------------------
Segment ID: 4
Raw Rule:   Prior_Bankruptcy=<ArrowStringArray>
['Yes']
Length: 1, 

In [4]:
builder.explain_feature_journey("Prior_Bankruptcy")

AUDIT TRAIL FOR FEATURE: 'Prior_Bankruptcy'

[Iteration 1]
  • Current dynamic IV   : 5.2008
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : Missed_Payment_Pattern=<ArrowStringArray>
['Frequent']
Length: 1, dtype: str (Variables: ['Missed_Payment_Pattern'])

[Iteration 2]
  • Current dynamic IV   : 5.9548
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  🎉 SELECTED as part of winning rule!
     Rule: Prior_Bankruptcy=<ArrowStringArray>
['Yes']
Length: 1, dtype: str & Housing_Status=<ArrowStringArray>
['Rent']
Length: 1, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
Length: 1, dtype: str

[Iteration 3]
  • Current dynamic IV   : 4.4531
  • Previous times used  : 1
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : Housing_Status=<ArrowStringArray>
['Rent']
Length: 1, dtype: str & Employment_Type=<ArrowStringArray>
['Unemployed']
L

In [5]:
# Preparing the dataset for scoring and decile banding.
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN Missed_Payment_Pattern IN ('Frequent') AND Housing_Status IN ('Other', 'Rent') AND Employment_Type IN ('Unemployed')
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN Missed_Payment_Pattern IN ('Frequent') AND Employment_Type IN ('Unemployed')
            THEN 1 ELSE 0 END AS seg_2,
            CASE WHEN Prior_Bankruptcy IN ('Yes')
            THEN 1 ELSE 0 END AS seg_3,
            FROM predicted
""").df()
conn.close()

In [6]:
scorer = StrategicSegmentScore(
    target_col="Default_Status",
    primary_key="Customer_ID",
    segment_cols=["seg_1", "seg_2", "seg_3"],
)

In [7]:
scorer.calculate_and_export_weights(predicted)

2026-07-11 17:00:03,590 | INFO     | [scorer.py:49] | Initializing out-of-core DuckDB scorecard engine...
2026-07-11 17:00:03,976 | INFO     | [scorer.py:85] | Computing harmonic scorecard weights...
2026-07-11 17:00:03,977 | INFO     | [scorer.py:114] | Scoring 100M rows natively via C++ database engine...
2026-07-11 17:00:04,052 | INFO     | [scorer.py:129] | Dataset Zero-Inflation Rate: 73.98%
2026-07-11 17:00:04,053 | INFO     | [scorer.py:132] | Calibrating deciles across active populations...


{'model_metadata': {'total_training_population': 250000,
  'active_scored_population': 250000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2602},
 'segment_weights': {'seg_1': {'weight': 15,
   'lift': 3.819,
   'response_rate': 0.9939,
   'capture_rate': 0.0199},
  'seg_2': {'weight': 38,
   'lift': 3.3868,
   'response_rate': 0.8814,
   'capture_rate': 0.0596},
  'seg_3': {'weight': 36,
   'lift': 1.479,
   'response_rate': 0.3849,
   'capture_rate': 0.1782}},
 'decile_min_thresholds': {'1': 89,
  '2': 36,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [8]:
scorecard_json_path = "/workspaces/Strategic_Segment_Builder/package/src/scorecard_model_2026_07_11_16_59_17.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-11 17:00:20,420 | INFO     | [3766329054.py:2] | Loading scorecard model artifact from /workspaces/Strategic_Segment_Builder/package/src/scorecard_model_2026_07_11_16_59_17.json...


In [9]:
model_artifact.get("decile_min_thresholds")

{'1': 89,
 '2': 36,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [11]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 15
Segment: seg_2 | Weight: 38
Segment: seg_3 | Weight: 36


In [12]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 15 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 38 ELSE 0 END AS seg_2_weighted,
    CASE WHEN seg_3 = 1 THEN 36 ELSE 0 END AS seg_3_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted + seg_3_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=89 THEN 1
                    WHEN total_weight >= 36 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [13]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(Default_Status) AS events, 
                    (SUM(Default_Status)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,1,174,174.0,100.000000
1,2,33801,14773.0,43.705808
2,3,216025,50114.0,23.198241
